In [0]:
import time
from pyspark.sql.functions import col

#setup context

catalog = "dev"
schema = "ecommerce_governed"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

target_user = 554748717
print("Query Plan (Unoptimized):")
spark.sql(f"SELECT * FROM silver_events WHERE user_id = {target_user}").explain()

In [0]:
table_optimized = "silver_events_optimized"

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {table_optimized}
          USING DELTA
          PARTITIONED BY (event_date)
          AS SELECT * FROM silver_events
          """)

# Apply Zorder to physically reorganize the parquet files to co-locate similar user_ids
spark.sql(f"OPTIMIZE {table_optimized} ZORDER BY (user_id, product_id)")



In [0]:
# Benchmarking

def benchmark_query(table_name, user_id):
    start_time = time.time()
    count = spark.sql(f"SELECT * from {table_name} WHERE user_id = {user_id}").count()
    duration = time.time() - start_time
    return duration, count

# Checking for Unoptimized table (Full Scan)
time_base , count_base = benchmark_query("silver_events", target_user)
print(f"Unoptimized Table : {time_base} seconds, {count_base} records")

# Checking for Optimized table (Partition Scan)

time_opt , count_opt = benchmark_query(table_optimized, target_user)
print(f"Optimized Table : {time_opt} seconds, {count_opt} records")


# Calculate Improvement
improvement = time_base / time_opt
print(f"\n🚀 Speedup Factor: {improvement:.1f}x Faster!")

